<a href="https://colab.research.google.com/github/Odeyiany2/ACCESS-6.0-Skills-Acquisition-Program-Data-Science/blob/main/week%205/Week_5_Session_10_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Week 5: Session 10 - Regression Modeling**

**Dataset:** Budgetwise Finance Dataset  
**Focus:** Bringing everything together — encoding, scaling, feature engineering, model building, and evaluation.


### **Learning Objectives**
- Understand linear regression concepts and assumptions  
- Build a regression model for prediction  
- Evaluate models using MAE and RMSE  
- Interpret coefficients and results

In [1]:
#libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error #metrics to evaluate our models performance

In [2]:
#getting the dataset
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/budgetwise_cleaned.csv')
df.head() # Display the first few rows

Mounted at /content/drive


,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,year,month,month_name,day_of_week,quarter,spending_level,is_high_value,is_weekend,total_user_spending,avg_user_transaction
0,T4999,U018,2023-04-25,Expense,education,3888,card,ahmedabad,2023,4,April,Tuesday,2,Medium,False,False,759875,8259.510870
1,T12828,U133,2021-12-16,Expense,rent,649,Unknown,hyderabad,2021,12,December,Thursday,4,Small,False,False,998527,12030.445783
2,T7403,U091,2021-12-16,Income,freelance,13239,Csh,bangalore,2021,12,December,Thursday,4,Large,True,False,1127019,11500.193878
3,T7495,U088,2021-12-16,Expense,entertainment,2287,CARD,hyderabad,2021,12,December,Thursday,4,Medium,False,False,1144448,12439.652174
4,T12465,U042,2021-12-16,Expense,food,4168,Unknown,not specified,2021,12,December,Thursday,4,Medium,False,False,1015065,11803.081395


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13755 entries, 0 to 13754
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   transaction_id        13755 non-null  object 
 1   user_id               13755 non-null  object 
 2   date                  13755 non-null  object 
 3   transaction_type      13755 non-null  object 
 4   category              13755 non-null  object 
 5   amount                13755 non-null  int64  
 6   payment_mode          13755 non-null  object 
 7   location              13755 non-null  object 
 8   year                  13755 non-null  int64  
 9   month                 13755 non-null  int64  
 10  month_name            13755 non-null  object 
 11  day_of_week           13755 non-null  object 
 12  quarter               13755 non-null  int64  
 13  spending_level        13755 non-null  object 
 14  is_high_value         13755 non-null  bool   
 15  is_weekend         

In [4]:
#statistics summary
df.describe().T

,count,mean,std,min,25%,50%,75%,max
amount,13755.0,1.210255e+04,17244.765771,0.00000,2797.500000,5.784000e+03,9.372500e+03,7.999900e+04
year,13755.0,2.021595e+03,1.020487,2021.00000,2021.000000,2.021000e+03,2.022000e+03,2.024000e+03
month,13755.0,9.842312e+00,3.454275,1.00000,8.000000,1.200000e+01,1.200000e+01,1.200000e+01
quarter,13755.0,3.413377e+00,1.013919,1.00000,3.000000,4.000000e+00,4.000000e+00,4.000000e+00
total_user_spending,13755.0,1.119609e+06,195890.690972,723238.00000,998527.000000,1.111502e+06,1.261749e+06,1.813038e+06
avg_user_transaction,13755.0,1.210255e+04,1822.250073,8259.51087,10851.829545,1.203279e+04,1.326523e+04,1.694428e+04


### **Define Target Variable and Features**

For this project, we assume:
- **Target Variable:** `y`
- **Features:** All other relevant columns `X`


In [5]:
#check columns
df.columns

Index(['transaction_id', 'user_id', 'date', 'transaction_type', 'category',
       'amount', 'payment_mode', 'location', 'year', 'month', 'month_name',
       'day_of_week', 'quarter', 'spending_level', 'is_high_value',
       'is_weekend', 'total_user_spending', 'avg_user_transaction'],
      dtype='object')

In [10]:
#defining X and y
y = df["amount"] #target variable
X = df.drop(columns = ["amount", "transaction_id", "user_id", "date"], axis = 1) #features

In [11]:
y

,amount
0,3888
1,649
2,13239
3,2287
4,4168
...,...
13750,9952
13751,7335
13752,4353
13753,1048


In [12]:
X.head()

,transaction_type,category,payment_mode,location,year,month,month_name,day_of_week,quarter,spending_level,is_high_value,is_weekend,total_user_spending,avg_user_transaction
0,Expense,education,card,ahmedabad,2023,4,April,Tuesday,2,Medium,False,False,759875,8259.510870
1,Expense,rent,Unknown,hyderabad,2021,12,December,Thursday,4,Small,False,False,998527,12030.445783
2,Income,freelance,Csh,bangalore,2021,12,December,Thursday,4,Large,True,False,1127019,11500.193878
3,Expense,entertainment,CARD,hyderabad,2021,12,December,Thursday,4,Medium,False,False,1144448,12439.652174
4,Expense,food,Unknown,not specified,2021,12,December,Thursday,4,Medium,False,False,1015065,11803.081395


### **Identify Categorical and Numerical Features**

This helps us apply the correct preprocessing steps.

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13755 entries, 0 to 13754
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   transaction_id        13755 non-null  object 
 1   user_id               13755 non-null  object 
 2   date                  13755 non-null  object 
 3   transaction_type      13755 non-null  object 
 4   category              13755 non-null  object 
 5   amount                13755 non-null  int64  
 6   payment_mode          13755 non-null  object 
 7   location              13755 non-null  object 
 8   year                  13755 non-null  int64  
 9   month                 13755 non-null  int64  
 10  month_name            13755 non-null  object 
 11  day_of_week           13755 non-null  object 
 12  quarter               13755 non-null  int64  
 13  spending_level        13755 non-null  object 
 14  is_high_value         13755 non-null  bool   
 15  is_weekend         

In [13]:
categorical_features = X.select_dtypes(include=["object"]).columns
numerical_features = X.select_dtypes(include = ["float64", "int64"]).columns


categorical_features, numerical_features

(Index(['transaction_type', 'category', 'payment_mode', 'location',
        'month_name', 'day_of_week', 'spending_level'],
       dtype='object'),
 Index(['year', 'month', 'quarter', 'total_user_spending',
        'avg_user_transaction'],
       dtype='object'))

### **Preprocessing: Encoding and Scaling**

- **Categorical features:** One-Hot Encoding  
- **Numerical features:** Standard Scaling


In [17]:
X.head()

,transaction_type,category,payment_mode,location,year,month,month_name,day_of_week,quarter,spending_level,is_high_value,is_weekend,total_user_spending,avg_user_transaction
0,Expense,education,card,ahmedabad,2023,4,April,Tuesday,2,Medium,False,False,759875,8259.510870
1,Expense,rent,Unknown,hyderabad,2021,12,December,Thursday,4,Small,False,False,998527,12030.445783
2,Income,freelance,Csh,bangalore,2021,12,December,Thursday,4,Large,True,False,1127019,11500.193878
3,Expense,entertainment,CARD,hyderabad,2021,12,December,Thursday,4,Medium,False,False,1144448,12439.652174
4,Expense,food,Unknown,not specified,2021,12,December,Thursday,4,Medium,False,False,1015065,11803.081395


In [18]:
#scaling
scale = StandardScaler()
encoder = OneHotEncoder(handle_unknown = "ignore")


#column transformer
preprocessor = ColumnTransformer(
    transformers = [
        ("num", scale, numerical_features),
        ("cat", encoder, categorical_features)
    ]
)


### **Train-Test Split**

We split the data to evaluate how well our model generalizes.

Typical split = 80/20, 70/30, 70/10/20


In [19]:
#split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)



In [20]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11004, 14), (2751, 14), (11004,), (2751,))

### **Build the Regression Model Pipeline**

Using a pipeline ensures clean and reproducible workflows.


Linear Regression -> explains the relationship between the inputs (features) and our output (target variable) using best fit line

amount
expense_category

y = mx+c

In [21]:
#initializiing the model
lr = LinearRegression()

model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("regressor", lr)
    ]
)
model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['year', 'month', 'quarter', 'total_user_spending',
       'avg_user_transaction'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['transaction_type', 'category', 'payment_mode', 'location',
       'month_name', 'day_of_week', 'spending_level'],
      dtype='object'))])),
                ('regressor', LinearRegression())])

### **Train the Model**


In [22]:
#model training
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['year', 'month', 'quarter', 'total_user_spending',
       'avg_user_transaction'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['transaction_type', 'category', 'payment_mode', 'location',
       'month_name', 'day_of_week', 'spending_level'],
      dtype='object'))])),
                ('regressor', LinearRegression())])

### **Make Predictions**

In [24]:
#make predictions
y_preds = model.predict(X_test)

### **Model Evaluation**

We evaluate performance using:
- **MAE (Mean Absolute Error)**
- **RMSE (Root Mean Squared Error)**


y_test(actual) = 5000
y_pred(model's prediction) = 4990

y_test(actual) = 6000
Y_pred(model's_prediction) = 5900

Absolute error = 4990 - 5000 = -10 = 10
Absolute error = 5900 - 6000 = -100 = 100

MAE = 100+10/2 = 110/2 = 55


RMSE = root mean squared error

[Metrics](https://scikit-learn.org/stable/modules/generated/sklearn.metrics)

[Linear Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)


In [25]:
#using mean absolute error
mae = mean_absolute_error(y_test, y_preds)
rmse = np.sqrt(mean_squared_error(y_test,y_preds))

#print the results
print("MAE:", mae)
print("RMSE:", rmse)

MAE: 5351.266202038089
RMSE: 9472.50950134109


In [27]:
#import r2 score
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_preds)
print("R2 Score:", r2)

R2 Score: 0.7001463826878978


,amount
0,3888
1,649
2,13239
3,2287
4,4168
...,...
13750,9952
13751,7335
13752,4353
13753,1048


### **Interpreting Model Coefficients**

Understanding how features influence predictions.


In [29]:
#get features names after preprocessing
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

#get coefficients
coefficients = model.named_steps["regressor"].coef_

#create a dataframe
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values(by = "Coefficient", ascending = False)


#get the top 10
coef_df.head(20)

,Feature,Coefficient
6,cat__transaction_type_Income,17002.086521
16,cat__category_rent,6082.207484
72,cat__spending_level_Large,5267.548370
7,cat__category_bonus,2576.664634
17,cat__category_salary,2236.612973
13,cat__category_investment,1813.752493
59,cat__month_name_June,807.295650
67,cat__day_of_week_Saturday,742.413160
58,cat__month_name_July,669.695219
62,cat__month_name_November,659.735769


In [ ]:
## Random Forest Regressor ->

## **Assignment**

Build a model for the Warehouse dataset and evaluate its performance.